Interactive Plots and Maps for the Exploratory Data Analysis

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [8]:
df = pd.read_csv(
    "customs_features.csv",
    parse_dates=["entry_date", "accepted_date", "release_date"],
    dtype={"hs_code": str},
)

In [9]:
df.head(3)

,shipment_id,entry_no,hs_code,product_desc,supplier,country_origin,country_destination,declared_value_usd,invoice_value_usd,weight_kg,...,slow_clearance,hs_chapter,hs_section,entry_day_of_week,entry_month,entry_is_weekend,license_missing,gap_pct,high_discrepancy,hs_section_name
0,SHIP000001,22EMKIM444597,303286,MEDICINAL PREPARATIONS,AgriTech Solutions,TANZANIA,UGANDA,88778.63,90696.42,3796.45,...,N,30,VI,2,7,False,N,0.021145,N,Chemical and pharmaceutical products
1,SHIP000002,22EMKIM486673,842519,CONVEYOR BELT,Industrial Supply Co,DRC,NIGERIA,40045.87,41736.99,2890.99,...,Y,84,XVI,3,11,False,N,0.040518,N,Machinery and electrical
2,SHIP000003,22EMKIM407331,302654,SURGICAL SUPPLIES,ChemCorp Industries,THAILAND,SOUTH AFRICA,55842.05,54005.62,3443.93,...,N,30,VI,5,6,True,N,-0.034004,N,Chemical and pharmaceutical products


In [10]:
df.columns

Index(['shipment_id', 'entry_no', 'hs_code', 'product_desc', 'supplier',
       'country_origin', 'country_destination', 'declared_value_usd',
       'invoice_value_usd', 'weight_kg', 'volume_m3', 'broker',
       'port_of_entry', 'entry_date', 'accepted_date', 'release_date',
       'duty_amount_usd', 'tax_amount_usd', 'broker_fee_usd', 'incoterm',
       'license_required', 'license_obtained', 'fta_claimed',
       'restricted_party_pass', 'container_number', 'slow_clearance',
       'hs_chapter', 'hs_section', 'entry_day_of_week', 'entry_month',
       'entry_is_weekend', 'license_missing', 'gap_pct', 'high_discrepancy',
       'hs_section_name'],
      dtype='str')

In [15]:
# CHART 9 (interactive): Slow-clearance rate by port
# Question: Which ports clear more slowly, and how many shipments back each rate?
# Method: Share of shipments with slow_clearance == "Y" per port; hover shows count.

overall = (df["slow_clearance"] == "Y").mean()

rates = (df.assign(flag=df["slow_clearance"] == "Y")
           .groupby("port_of_entry")
           .agg(rows=("shipment_id", "count"), rate=("flag", "mean"))
           .sort_values("rate", ascending=False)
           .reset_index())

fig = px.bar(rates, x="rate", y="port_of_entry", orientation="h",
             hover_data=["rows"],
             title="Slow-clearance rate by port")
fig.add_vline(x=overall, line_dash="dash", line_color="red")
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  xaxis_title="Share of shipments flagged (slow_clearance)",
                  yaxis_title="")
fig.show()

rates

,port_of_entry,rows,rate
0,ALEXANDRIA,701,0.335235
1,MOMBASA,719,0.307371
2,LAGOS,701,0.306705
3,CASABLANCA,743,0.301480
4,MOMBASA ICD,231,0.207792
5,DURBAN,255,0.196078
6,TEMA,258,0.193798
7,DAR ES SALAAM,227,0.189427
8,PORT ELIZABETH,239,0.188285
9,NAIROBI,203,0.182266


In [16]:
# STEP 2a: Origin countries and shipment counts
# Question: What exact spellings are in country_origin, so the ISO-3 lookup can map all of them?
# Method: Count shipments per origin and print every name as it appears.

origins = (df.groupby("country_origin")
             .agg(rows=("shipment_id", "count"))
             .sort_values("rows", ascending=False)
             .reset_index())

print(origins.to_string())               # to_string shows all rows
print(sorted(df["country_origin"].unique()))

   country_origin  rows
0           INDIA   421
1         VIETNAM   398
2        THAILAND   393
3           CHINA   392
4          TURKEY   387
5         GERMANY   149
6        MALAYSIA   131
7     SOUTH KOREA   123
8             UAE   123
9             USA   122
10   COTE DIVOIRE   120
11        MOROCCO   119
12         ANGOLA   118
13     MOZAMBIQUE   118
14          GHANA   116
15       TANZANIA   115
16       ETHIOPIA   115
17          KENYA   114
18         UGANDA   113
19         ZAMBIA   113
20        NIGERIA   112
21             UK   111
22          JAPAN   111
23       CAMEROON   111
24            DRC   111
25   SOUTH AFRICA   109
26        SENEGAL   104
27         RWANDA    96
28          EGYPT    81
['ANGOLA', 'CAMEROON', 'CHINA', 'COTE DIVOIRE', 'DRC', 'EGYPT', 'ETHIOPIA', 'GERMANY', 'GHANA', 'INDIA', 'JAPAN', 'KENYA', 'MALAYSIA', 'MOROCCO', 'MOZAMBIQUE', 'NIGERIA', 'RWANDA', 'SENEGAL', 'SOUTH AFRICA', 'SOUTH KOREA', 'TANZANIA', 'THAILAND', 'TURKEY', 'UAE', 'UGANDA', 'UK', 

In [17]:
# STEP 2b: Map origin names to ISO-3 codes for the choropleth
# Question: Does every origin country have a code? Unmapped rows vanish from maps silently.
# Method: Dictionary lookup, then list any origin left without a code.

iso3 = {
    "INDIA": "IND", "VIETNAM": "VNM", "THAILAND": "THA", "CHINA": "CHN",
    "TURKEY": "TUR", "GERMANY": "DEU", "MALAYSIA": "MYS", "SOUTH KOREA": "KOR",
    "UAE": "ARE", "USA": "USA", "COTE DIVOIRE": "CIV", "MOROCCO": "MAR",
    "ANGOLA": "AGO", "MOZAMBIQUE": "MOZ", "GHANA": "GHA", "TANZANIA": "TZA",
    "ETHIOPIA": "ETH", "KENYA": "KEN", "UGANDA": "UGA", "ZAMBIA": "ZMB",
    "NIGERIA": "NGA", "UK": "GBR", "JAPAN": "JPN", "CAMEROON": "CMR",
    "DRC": "COD", "SOUTH AFRICA": "ZAF", "SENEGAL": "SEN", "RWANDA": "RWA",
    "EGYPT": "EGY",
}

origins["iso3"] = origins["country_origin"].map(iso3)

print("Unmapped:", origins.loc[origins["iso3"].isna(), "country_origin"].tolist())
print("Codes of wrong length:", origins.loc[origins["iso3"].str.len() != 3, "country_origin"].tolist())
print("Duplicate codes:", origins["iso3"].duplicated().sum())

Unmapped: []
Codes of wrong length: []
Duplicate codes: 0


In [18]:
ref = px.data.gapminder()[["country", "iso_alpha"]].drop_duplicates()
not_in_ref = origins.loc[~origins["iso3"].isin(ref["iso_alpha"]), ["country_origin", "iso3"]]
not_in_ref

,country_origin,iso3
8,UAE,ARE


In [23]:
# CHART 10 (interactive): Origin choropleths
# Question 1: Which origin countries have the highest slow-clearance rates?
# Question 2: Which origin countries have the highest high-discrepancy rates?
# Method: Share of shipments flagged per origin, shaded on a world map; hover shows count.

origin_rates = (df.assign(slow=df["slow_clearance"] == "Y",
                          discrepancy=df["high_discrepancy"] == "Y")
                  .groupby("country_origin")
                  .agg(rows=("shipment_id", "count"),
                       slow_rate=("slow", "mean"),
                       discrepancy_rate=("discrepancy", "mean"))
                  .reset_index())
origin_rates["iso3"] = origin_rates["country_origin"].map(iso3)

assert origin_rates["iso3"].notna().all()     # no origin left off the map

def origin_map(rate_col, title):
    fig = px.choropleth(origin_rates, locations="iso3", color=rate_col,
                        hover_name="country_origin", hover_data=["rows"],
                        color_continuous_scale="Reds",
                        range_color=(0, 0.35),
                        title=title)
    fig.show()
    return fig

fig_slow = origin_map("slow_rate", "Slow-clearance rate by origin")
fig_disc = origin_map("discrepancy_rate", "High-discrepancy rate by origin")

origin_rates.sort_values("discrepancy_rate", ascending=False)

,country_origin,rows,slow_rate,discrepancy_rate,iso3
2,CHINA,392,0.308673,0.260204,CHN
27,VIETNAM,398,0.319095,0.223618,VNM
22,TURKEY,387,0.286822,0.219638,TUR
21,THAILAND,393,0.269720,0.206107,THA
23,UAE,123,0.284553,0.195122,ARE
6,ETHIOPIA,115,0.304348,0.191304,ETH
12,MALAYSIA,131,0.236641,0.175573,MYS
18,SOUTH AFRICA,109,0.183486,0.174312,ZAF
9,INDIA,421,0.315914,0.173397,IND
8,GHANA,116,0.172414,0.172414,GHA


In [24]:
origin_rates[["slow_rate", "discrepancy_rate"]].max()

slow_rate           0.319095
discrepancy_rate    0.260204
dtype: float64